# Golem-T13: Hybrid MLP + Decision Tree Classifier

Complete workflow demonstrating hybrid approach where MLP learns embeddings used by tree classifiers.

In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, str(Path.cwd() / 'src'))

from golem import MLPClassifier, MyDecisionTreeClassifier, MyRandomForestClassifier, Pipeline

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

### defining functions

In [2]:
def prepare_dataset(n_samples: int = 2000, noise: float = 0.4, batch_size: int = 128):
    pipeline = Pipeline(n=n_samples, noise=noise, seed=SEED)
    X_raw, y_raw = pipeline.X, pipeline.y
    print(f"Dataset created: {n_samples} samples, noise={noise}")
    return X_raw, y_raw

In [3]:
def create_dataloaders(X_raw, y_raw, batch_size: int = 128):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=SEED
    )
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
    return train_loader, val_loader, test_loader, X_train, y_train, X_test, y_test

In [4]:
def initialize_mlp_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    model = MLPClassifier(depth=7, input_dim=2, hidden_dim=64, out_dim=1)
    model.to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    epochs = 50
    print(f"Training for {epochs} epochs...")
    return model, loss_fn, optimizer, device, epochs

In [5]:
def train_mlp(model, train_loader, val_loader, loss_fn, optimizer, device, epochs):
    for epoch in range(epochs):
        model.train()
        for batch, (X, y) in enumerate(train_loader):
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        if epoch % 10 == 0:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for X, y in val_loader:
                    X, y = X.to(device), y.to(device)
                    pred = model(X)
                    val_loss += loss_fn(pred, y).item()
            print(f"Epoch {epoch}: Val Loss: {val_loss/len(val_loader):.4f}")

In [6]:
def evaluate_mlp(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            pred_labels = (torch.sigmoid(pred) > 0.5).float()
            correct += (pred_labels == y).sum().item()
            total += y.size(0)
    mlp_accuracy = correct / total
    print(f"MLP Test Accuracy: {mlp_accuracy*100:.2f}%")
    return mlp_accuracy

In [7]:
def extract_embeddings(model, train_loader, val_loader, test_loader, device):
    X_train_emb, y_train_emb = model.extract(train_loader, device)
    X_val_emb, y_val_emb = model.extract(val_loader, device)
    X_test_emb, y_test_emb = model.extract(test_loader, device)
    print(f"Embeddings extracted:")
    print(f"  Train: {X_train_emb.shape}")
    print(f"  Val:   {X_val_emb.shape}")
    print(f"  Test:  {X_test_emb.shape}")
    return X_train_emb, y_train_emb, X_val_emb, y_val_emb, X_test_emb, y_test_emb

In [8]:
def initialize_comparison(X_train_raw_split, y_train_raw_split, X_test_raw_split, y_test_raw_split, mlp_accuracy):
    results = []
    results.append({
        'Model': 'MLP Solo',
        'Data_Type': 'Raw',
        'Accuracy': mlp_accuracy
    })
    print("1. MLP Solo")
    print(f"   Accuracy: {mlp_accuracy*100:.2f}%")
    return results

In [9]:
def train_decision_tree_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split):
    dt_solo = MyDecisionTreeClassifier(max_depth=10, min_samples_split=5)
    dt_solo.fit(X_train_raw_split, y_train_raw_split)
    dt_solo_acc = accuracy_score(y_test_raw_split, dt_solo.predict(X_test_raw_split))
    return dt_solo, dt_solo_acc

In [10]:
def train_decision_tree_sklearn(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split):
    dt_sklearn = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=SEED)
    dt_sklearn.fit(X_train_raw_split, y_train_raw_split)
    dt_sklearn_acc = accuracy_score(y_test_raw_split, dt_sklearn.predict(X_test_raw_split))
    return dt_sklearn, dt_sklearn_acc

In [11]:
def train_random_forest_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split):
    rf_solo = MyRandomForestClassifier(n_estimators=10, max_depth=10, min_samples_split=5)
    rf_solo.fit(X_train_raw_split, y_train_raw_split)
    rf_solo_acc = accuracy_score(y_test_raw_split, rf_solo.predict(X_test_raw_split))
    return rf_solo, rf_solo_acc

In [12]:
def train_random_forest_sklearn(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split):
    rf_sklearn = RandomForestClassifier(n_estimators=10, max_depth=10, min_samples_split=5, random_state=SEED)
    rf_sklearn.fit(X_train_raw_split, y_train_raw_split)
    rf_sklearn_acc = accuracy_score(y_test_raw_split, rf_sklearn.predict(X_test_raw_split))
    return rf_sklearn, rf_sklearn_acc

In [13]:
def train_gradient_boosting_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split):
    gb_solo = GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=SEED)
    gb_solo.fit(X_train_raw_split, y_train_raw_split)
    gb_solo_acc = accuracy_score(y_test_raw_split, gb_solo.predict(X_test_raw_split))
    return gb_solo, gb_solo_acc

In [14]:
def train_hybrid_decision_tree(X_train_emb, y_train_emb, X_test_emb, y_test_emb):
    dt_hybrid = MyDecisionTreeClassifier(max_depth=10, min_samples_split=5)
    dt_hybrid.fit(X_train_emb, y_train_emb)
    dt_hybrid_acc = accuracy_score(y_test_emb, dt_hybrid.predict(X_test_emb))
    return dt_hybrid, dt_hybrid_acc

In [15]:
def train_hybrid_decision_tree_sklearn(X_train_emb, y_train_emb, X_test_emb, y_test_emb):
    dt_hybrid_sklearn = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=SEED)
    dt_hybrid_sklearn.fit(X_train_emb, y_train_emb)
    dt_hybrid_sklearn_acc = accuracy_score(y_test_emb, dt_hybrid_sklearn.predict(X_test_emb))
    return dt_hybrid_sklearn, dt_hybrid_sklearn_acc

In [16]:
def train_hybrid_random_forest(X_train_emb, y_train_emb, X_test_emb, y_test_emb):
    rf_hybrid = MyRandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5)
    rf_hybrid.fit(X_train_emb, y_train_emb)
    rf_hybrid_acc = accuracy_score(y_test_emb, rf_hybrid.predict(X_test_emb))
    return rf_hybrid, rf_hybrid_acc

In [17]:
def train_hybrid_random_forest_sklearn(X_train_emb, y_train_emb, X_test_emb, y_test_emb):
    rf_hybrid_sklearn = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=5, random_state=SEED)
    rf_hybrid_sklearn.fit(X_train_emb, y_train_emb)
    rf_hybrid_sklearn_acc = accuracy_score(y_test_emb, rf_hybrid_sklearn.predict(X_test_emb))
    return rf_hybrid_sklearn, rf_hybrid_sklearn_acc

In [18]:
def train_hybrid_gradient_boosting(X_train_emb, y_train_emb, X_test_emb, y_test_emb):
    gb_hybrid = GradientBoostingClassifier(n_estimators=50, max_depth=5, random_state=SEED)
    gb_hybrid.fit(X_train_emb, y_train_emb)
    gb_hybrid_acc = accuracy_score(y_test_emb, gb_hybrid.predict(X_test_emb))
    return gb_hybrid, gb_hybrid_acc

In [19]:
def objective_random_forest_sklearn(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': SEED
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [20]:
def objective_random_forest_custom(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 5, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
    }
    model = MyRandomForestClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [21]:
def objective_gradient_boosting(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 200),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': SEED
    }
    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [22]:
def objective_decision_tree_sklearn(trial, X_train, y_train, X_val, y_val):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': SEED
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [23]:
def objective_decision_tree_custom(trial, X_train, y_train, X_val, y_val):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
    }
    model = MyDecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [ ]:
def objective_hybrid_decision_tree_sklearn(trial, X_train, y_train, X_val, y_val):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'random_state': SEED
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [ ]:
def objective_hybrid_decision_tree_custom(trial, X_train, y_train, X_val, y_val):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
    }
    model = MyDecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [ ]:
def objective_hybrid_random_forest_sklearn(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 10, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'random_state': SEED
    }
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [ ]:
def objective_hybrid_gradient_boosting(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 200),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'random_state': SEED
    }
    model = GradientBoostingClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [24]:
def display_results(results):
    df_results = pd.DataFrame(results)
    print("\n" + "="*70)
    print("FINAL RESULTS")
    print("="*70 + "\n")
    print(df_results.to_string(index=False))
    best_idx = df_results['Accuracy'].idxmax()
    return df_results

In [ ]:
def display_final_results_with_optimization(results):
    rf_optimized_acc = globals().get('rf_optimized_acc', np.nan)
    gb_optimized_acc = globals().get('gb_optimized_acc', np.nan)
    dt_optimized_acc = globals().get('dt_optimized_acc', np.nan)
    dt_custom_optimized_acc = globals().get('dt_custom_optimized_acc', np.nan)
    dt_hybrid_sklearn_optimized_acc = globals().get('dt_hybrid_sklearn_optimized_acc', np.nan)
    dt_hybrid_custom_optimized_acc = globals().get('dt_hybrid_custom_optimized_acc', np.nan)
    rf_hybrid_sklearn_optimized_acc = globals().get('rf_hybrid_sklearn_optimized_acc', np.nan)
    gb_hybrid_optimized_acc = globals().get('gb_hybrid_optimized_acc', np.nan)

    final_results = results.copy()
    final_results.append({'Model': 'RandomForest sklearn (optimized)', 'Data_Type': 'Raw', 'Accuracy': rf_optimized_acc})
    final_results.append({'Model': 'GradientBoosting (optimized)', 'Data_Type': 'Raw', 'Accuracy': gb_optimized_acc})
    final_results.append({'Model': 'DecisionTree sklearn (optimized)', 'Data_Type': 'Raw', 'Accuracy': dt_optimized_acc})
    final_results.append({'Model': 'DecisionTree Solo (optimized)', 'Data_Type': 'Raw', 'Accuracy': dt_custom_optimized_acc})
    final_results.append({'Model': 'HYBRID: MLP->Tree sklearn (optimized)', 'Data_Type': 'Embeddings', 'Accuracy': dt_hybrid_sklearn_optimized_acc})
    final_results.append({'Model': 'HYBRID: MLP->Tree (optimized)', 'Data_Type': 'Embeddings', 'Accuracy': dt_hybrid_custom_optimized_acc})
    final_results.append({'Model': 'HYBRID: MLP->RandomForest sklearn (optimized)', 'Data_Type': 'Embeddings', 'Accuracy': rf_hybrid_sklearn_optimized_acc})
    final_results.append({'Model': 'HYBRID: GradientBoosting (optimized)', 'Data_Type': 'Embeddings', 'Accuracy': gb_hybrid_optimized_acc})

    df_final = pd.DataFrame(final_results)

    print("\n" + "="*80)
    print("FINAL RESULTS WITH OPTIMIZATION")
    print("="*80 + "\n")
    print(df_final.to_string(index=False))

    df_valid = df_final[df_final['Accuracy'].notna()]
    if not df_valid.empty:
        best_idx = df_valid['Accuracy'].idxmax()
        best_row = df_valid.loc[best_idx]
        print("\nBest model:")
        print(f"  {best_row['Model']} ({best_row['Data_Type']}) -> {best_row['Accuracy']*100:.2f}%")
    else:
        print("\nNo optimized results available (all NaN).")

    return df_final

### Results

In [25]:
X_raw, y_raw = prepare_dataset()

Dataset created: 2000 samples, noise=0.4


In [26]:
train_loader, val_loader, test_loader, X_train_raw_split, y_train_raw_split, X_test_raw_split, y_test_raw_split = create_dataloaders(X_raw, y_raw)

Train: 1600, Val: 200, Test: 200


In [27]:
model, loss_fn, optimizer, device, epochs = initialize_mlp_model()

Using device: cpu
Training for 50 epochs...


In [28]:
train_mlp(model, train_loader, val_loader, loss_fn, optimizer, device, epochs)

Epoch 0: Val Loss: 0.6823
Epoch 10: Val Loss: 0.3517
Epoch 20: Val Loss: 0.3347
Epoch 30: Val Loss: 0.3503
Epoch 40: Val Loss: 0.3385


In [29]:
mlp_accuracy = evaluate_mlp(model, test_loader, device)

MLP Test Accuracy: 88.50%


In [30]:
X_train_emb, y_train_emb, X_val_emb, y_val_emb, X_test_emb, y_test_emb = extract_embeddings(
    model, train_loader, val_loader, test_loader, device
)

Embeddings extracted:
  Train: (1600, 32)
  Val:   (200, 32)
  Test:  (200, 32)


In [31]:
results = initialize_comparison(
    X_train_raw_split, y_train_raw_split, X_test_raw_split, y_test_raw_split, mlp_accuracy
)

1. MLP Solo
   Accuracy: 88.50%


In [32]:
dt_solo, dt_solo_acc = train_decision_tree_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split)
results.append({'Model': 'DecisionTree Solo', 'Data_Type': 'Raw', 'Accuracy': dt_solo_acc})
print('2. Decision Tree Solo')
print(f'   Accuracy: {dt_solo_acc*100:.2f}%')

2. Decision Tree Solo
   Accuracy: 82.50%


In [33]:
dt_sklearn, dt_sklearn_acc = train_decision_tree_sklearn(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split)
results.append({'Model': 'DecisionTree sklearn', 'Data_Type': 'Raw', 'Accuracy': dt_sklearn_acc})
print('3. Decision Tree sklearn')
print(f'   Accuracy: {dt_sklearn_acc*100:.2f}%')

3. Decision Tree sklearn
   Accuracy: 80.00%


In [34]:
rf_solo, rf_solo_acc = train_random_forest_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split)
results.append({'Model': 'RandomForest Solo', 'Data_Type': 'Raw', 'Accuracy': rf_solo_acc})
print('4. Random Forest Solo')
print(f'   Accuracy: {rf_solo_acc*100:.2f}%')

4. Random Forest Solo
   Accuracy: 84.50%


In [35]:
rf_sklearn, rf_sklearn_acc = train_random_forest_sklearn(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split)
results.append({'Model': 'RandomForest sklearn', 'Data_Type': 'Raw', 'Accuracy': rf_sklearn_acc})
print('5. Random Forest sklearn')
print(f'   Accuracy: {rf_sklearn_acc*100:.2f}%')

5. Random Forest sklearn
   Accuracy: 86.50%


In [36]:
gb_solo, gb_solo_acc = train_gradient_boosting_solo(X_train_raw_split, X_test_raw_split, y_train_raw_split, y_test_raw_split)
results.append({'Model': 'GradientBoosting Solo', 'Data_Type': 'Raw', 'Accuracy': gb_solo_acc})
print('6. GradientBoosting Solo')
print(f'   Accuracy: {gb_solo_acc*100:.2f}%')

6. GradientBoosting Solo
   Accuracy: 85.00%


In [37]:
dt_hybrid, dt_hybrid_acc = train_hybrid_decision_tree(X_train_emb, y_train_emb, X_test_emb, y_test_emb)
results.append({'Model': 'HYBRID: MLP->Tree', 'Data_Type': 'Embeddings', 'Accuracy': dt_hybrid_acc})
print('7. HYBRID: MLP -> Decision Tree')
print(f'   Accuracy: {dt_hybrid_acc*100:.2f}%')

7. HYBRID: MLP -> Decision Tree
   Accuracy: 81.50%


In [38]:
dt_hybrid_sklearn, dt_hybrid_sklearn_acc = train_hybrid_decision_tree_sklearn(X_train_emb, y_train_emb, X_test_emb, y_test_emb)
results.append({'Model': 'HYBRID: MLP->Tree sklearn', 'Data_Type': 'Embeddings', 'Accuracy': dt_hybrid_sklearn_acc})
print('8. HYBRID: MLP -> Decision Tree sklearn')
print(f'   Accuracy: {dt_hybrid_sklearn_acc*100:.2f}%')

8. HYBRID: MLP -> Decision Tree sklearn
   Accuracy: 82.50%


In [39]:
rf_hybrid, rf_hybrid_acc = train_hybrid_random_forest(X_train_emb, y_train_emb, X_test_emb, y_test_emb)
results.append({'Model': 'HYBRID: MLP->RandomForest', 'Data_Type': 'Embeddings', 'Accuracy': rf_hybrid_acc})
print('9. HYBRID: MLP -> Random Forest')
print(f'   Accuracy: {rf_hybrid_acc*100:.2f}%')

9. HYBRID: MLP -> Random Forest
   Accuracy: 84.50%


In [40]:
rf_hybrid_sklearn, rf_hybrid_sklearn_acc = train_hybrid_random_forest_sklearn(X_train_emb, y_train_emb, X_test_emb, y_test_emb)
results.append({'Model': 'HYBRID: MLP->RandomForest sklearn', 'Data_Type': 'Embeddings', 'Accuracy': rf_hybrid_sklearn_acc})
print('10. HYBRID: MLP -> Random Forest sklearn')
print(f'   Accuracy: {rf_hybrid_sklearn_acc*100:.2f}%')

10. HYBRID: MLP -> Random Forest sklearn
   Accuracy: 84.50%


In [41]:
gb_hybrid, gb_hybrid_acc = train_hybrid_gradient_boosting(X_train_emb, y_train_emb, X_test_emb, y_test_emb)
results.append({'Model': 'GradientBoosting (Embeddings)', 'Data_Type': 'Embeddings', 'Accuracy': gb_hybrid_acc})
print('11. GradientBoosting on Embeddings')
print(f'   Accuracy: {gb_hybrid_acc*100:.2f}%')

11. GradientBoosting on Embeddings
   Accuracy: 86.00%


In [42]:
df_results = display_results(results)


FINAL RESULTS

                            Model  Data_Type  Accuracy
                         MLP Solo        Raw     0.885
                DecisionTree Solo        Raw     0.825
             DecisionTree sklearn        Raw     0.800
                RandomForest Solo        Raw     0.845
             RandomForest sklearn        Raw     0.865
            GradientBoosting Solo        Raw     0.850
                HYBRID: MLP->Tree Embeddings     0.815
        HYBRID: MLP->Tree sklearn Embeddings     0.825
        HYBRID: MLP->RandomForest Embeddings     0.845
HYBRID: MLP->RandomForest sklearn Embeddings     0.845
    GradientBoosting (Embeddings) Embeddings     0.860

WINNER: MLP Solo with accuracy 88.50%


## Optimization

In [43]:
sampler = TPESampler(seed=SEED)
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_train_raw_split, y_train_raw_split, test_size=0.2, random_state=SEED
)

In [45]:
print('Optimizing Random Forest (sklearn)')
study_rf = optuna.create_study(direction='maximize', sampler=sampler)
study_rf.optimize(
    lambda trial: objective_random_forest_sklearn(trial, X_train_opt, y_train_opt, X_val_opt, y_val_opt),
    n_trials=50,
    show_progress_bar=False
)

rf_optimized = RandomForestClassifier(**study_rf.best_params)
rf_optimized.fit(X_train_opt, y_train_opt)
rf_optimized_acc = accuracy_score(y_test_raw_split, rf_optimized.predict(X_test_raw_split))
print(f'Test accuracy: {rf_optimized_acc*100:.2f}%')
print(f'Standard RF test accuracy: {rf_sklearn_acc*100:.2f}%')

Optimizing Random Forest (sklearn)
Test accuracy: 85.50%
Standard RF test accuracy: 86.50%


### Optimization on Embeddings

In [54]:
X_train_emb_opt, X_val_emb_opt, y_train_emb_opt, y_val_emb_opt = train_test_split(
    X_train_emb, y_train_emb, test_size=0.2, random_state=SEED
)

In [56]:
print('Optimizing HYBRID Decision Tree (sklearn) on Embeddings')
study_dt_hybrid_sklearn = optuna.create_study(direction='maximize', sampler=sampler)
study_dt_hybrid_sklearn.optimize(
    lambda trial: objective_decision_tree_sklearn(trial, X_train_emb_opt, y_train_emb_opt, X_val_emb_opt, y_val_emb_opt),
    n_trials=50,
    show_progress_bar=False
)
dt_hybrid_sklearn_optimized = DecisionTreeClassifier(**study_dt_hybrid_sklearn.best_params)
dt_hybrid_sklearn_optimized.fit(X_train_emb_opt, y_train_emb_opt)
dt_hybrid_sklearn_optimized_acc = accuracy_score(y_test_emb, dt_hybrid_sklearn_optimized.predict(X_test_emb))
print(f'Test accuracy: {dt_hybrid_sklearn_optimized_acc*100:.2f}%')
print(f'Standard hybrid DT sklearn test accuracy: {dt_hybrid_sklearn_acc*100:.2f}%')

Optimizing HYBRID Decision Tree (sklearn) on Embeddings
Test accuracy: 87.00%
Standard hybrid DT sklearn test accuracy: 82.50%


In [59]:
print('Optimizing HYBRID Decision Tree (custom) on Embeddings')
study_dt_hybrid_custom = optuna.create_study(direction='maximize', sampler=sampler)
study_dt_hybrid_custom.optimize(
    lambda trial: objective_decision_tree_custom(trial, X_train_emb_opt, y_train_emb_opt, X_val_emb_opt, y_val_emb_opt),
    n_trials=50,
    show_progress_bar=False
)
dt_hybrid_custom_optimized = MyDecisionTreeClassifier(**study_dt_hybrid_custom.best_params)
dt_hybrid_custom_optimized.fit(X_train_emb_opt, y_train_emb_opt)
dt_hybrid_custom_optimized_acc = accuracy_score(y_test_emb, dt_hybrid_custom_optimized.predict(X_test_emb))
print(f'Test accuracy: {dt_hybrid_custom_optimized_acc*100:.2f}%')
print(f'Standard hybrid DT custom test accuracy: {dt_hybrid_acc*100:.2f}%')

Optimizing HYBRID Decision Tree (custom) on Embeddings
Test accuracy: 87.00%
Standard hybrid DT custom test accuracy: 81.50%


In [60]:
print('Optimizing HYBRID Random Forest (sklearn) on Embeddings')
study_rf_hybrid_sklearn = optuna.create_study(direction='maximize', sampler=sampler)
study_rf_hybrid_sklearn.optimize(
    lambda trial: objective_random_forest_sklearn(trial, X_train_emb_opt, y_train_emb_opt, X_val_emb_opt, y_val_emb_opt),
    n_trials=20,
    show_progress_bar=False
)
rf_hybrid_sklearn_optimized = RandomForestClassifier(**study_rf_hybrid_sklearn.best_params)
rf_hybrid_sklearn_optimized.fit(X_train_emb_opt, y_train_emb_opt)
rf_hybrid_sklearn_optimized_acc = accuracy_score(y_test_emb, rf_hybrid_sklearn_optimized.predict(X_test_emb))
print(f'Test accuracy: {rf_hybrid_sklearn_optimized_acc*100:.2f}%')
print(f'Standard hybrid RF sklearn test accuracy: {rf_hybrid_sklearn_acc*100:.2f}%')

Optimizing HYBRID Random Forest (sklearn) on Embeddings
Test accuracy: 87.50%
Standard hybrid RF sklearn test accuracy: 84.50%


In [61]:
print('Optimizing HYBRID Gradient Boosting on Embeddings')
study_gb_hybrid = optuna.create_study(direction='maximize', sampler=sampler)
study_gb_hybrid.optimize(
    lambda trial: objective_gradient_boosting(trial, X_train_emb_opt, y_train_emb_opt, X_val_emb_opt, y_val_emb_opt),
    n_trials=50,
    show_progress_bar=False
)
gb_hybrid_optimized = GradientBoostingClassifier(**study_gb_hybrid.best_params)
gb_hybrid_optimized.fit(X_train_emb_opt, y_train_emb_opt)
gb_hybrid_optimized_acc = accuracy_score(y_test_emb, gb_hybrid_optimized.predict(X_test_emb))
print(f'Test accuracy: {gb_hybrid_optimized_acc*100:.2f}%')
print(f'Standard hybrid GB test accuracy: {gb_hybrid_acc*100:.2f}%')

Optimizing HYBRID Gradient Boosting on Embeddings
Test accuracy: 87.50%
Standard hybrid GB test accuracy: 86.00%


In [46]:
# print('Optimizing Random Forest (custom)')
# study_rf_custom = optuna.create_study(direction='maximize', sampler=sampler)
# study_rf_custom.optimize(
#     lambda trial: objective_random_forest_custom(trial, X_train_opt, y_train_opt, X_val_opt, y_val_opt),
#     n_trials=20,
#     show_progress_bar=False
# )

# rf_custom_optimized = MyRandomForestClassifier(**study_rf_custom.best_params)
# rf_custom_optimized.fit(X_train_opt, y_train_opt)
# rf_custom_optimized_acc = accuracy_score(y_test_raw_split, rf_custom_optimized.predict(X_test_raw_split))
# print(f'Test accuracy: {rf_custom_optimized_acc*100:.2f}%')
# print(f'Standard custom RF test accuracy: {rf_solo_acc*100:.2f}%')


In [47]:
print('Optimizing Gradient Boosting')
study_gb = optuna.create_study(direction='maximize', sampler=sampler)
study_gb.optimize(
    lambda trial: objective_gradient_boosting(trial, X_train_opt, y_train_opt, X_val_opt, y_val_opt),
    n_trials=50,
    show_progress_bar=False
)
gb_optimized = GradientBoostingClassifier(**study_gb.best_params)
gb_optimized.fit(X_train_opt, y_train_opt)
gb_optimized_acc = accuracy_score(y_test_raw_split, gb_optimized.predict(X_test_raw_split))
print(f'Test accuracy: {gb_optimized_acc*100:.2f}%')
print(f'Standard GB test accuracy: {gb_solo_acc*100:.2f}%')


Optimizing Gradient Boosting
Test accuracy: 82.50%
Standard GB test accuracy: 85.00%


In [48]:
print('Optimizing Decision Tree (sklearn)')
study_dt = optuna.create_study(direction='maximize', sampler=sampler)
study_dt.optimize(
    lambda trial: objective_decision_tree_sklearn(trial, X_train_opt, y_train_opt, X_val_opt, y_val_opt),
    n_trials=50,
    show_progress_bar=False
)
dt_optimized = DecisionTreeClassifier(**study_dt.best_params)
dt_optimized.fit(X_train_opt, y_train_opt)
dt_optimized_acc = accuracy_score(y_test_raw_split, dt_optimized.predict(X_test_raw_split))
print(f'Test accuracy: {dt_optimized_acc*100:.2f}%')
print(f'Standard DT test accuracy: {dt_sklearn_acc*100:.2f}%')


Optimizing Decision Tree (sklearn)
Test accuracy: 85.50%
Standard DT test accuracy: 80.00%


In [49]:
print('Optimizing Decision Tree (custom)')
study_dt_custom = optuna.create_study(direction='maximize', sampler=sampler)
study_dt_custom.optimize(
    lambda trial: objective_decision_tree_custom(trial, X_train_opt, y_train_opt, X_val_opt, y_val_opt),
    n_trials=50,
    show_progress_bar=False
)


dt_custom_optimized = MyDecisionTreeClassifier(**study_dt_custom.best_params)
dt_custom_optimized.fit(X_train_opt, y_train_opt)
dt_custom_optimized_acc = accuracy_score(y_test_raw_split, dt_custom_optimized.predict(X_test_raw_split))
print(f'Test accuracy: {dt_custom_optimized_acc*100:.2f}%')
print(f'Standard custom DT test accuracy: {dt_solo_acc*100:.2f}%')


Optimizing Decision Tree (custom)
Best hyperparameters: {'max_depth': 8, 'min_samples_split': 2}
Best validation accuracy: 84.06%
Test accuracy: 84.50%
Standard custom DT test accuracy: 82.50%


In [62]:
df_final_results = display_final_results_with_optimization(results)


FINAL RESULTS WITH OPTIMIZATION

                            Model  Data_Type  Accuracy
                         MLP Solo        Raw     0.885
                DecisionTree Solo        Raw     0.825
             DecisionTree sklearn        Raw     0.800
                RandomForest Solo        Raw     0.845
             RandomForest sklearn        Raw     0.865
            GradientBoosting Solo        Raw     0.850
                HYBRID: MLP->Tree Embeddings     0.815
        HYBRID: MLP->Tree sklearn Embeddings     0.825
        HYBRID: MLP->RandomForest Embeddings     0.845
HYBRID: MLP->RandomForest sklearn Embeddings     0.845
    GradientBoosting (Embeddings) Embeddings     0.860
 RandomForest sklearn (optimized)        Raw     0.855
     GradientBoosting (optimized)        Raw     0.825
 DecisionTree sklearn (optimized)        Raw     0.855
    DecisionTree Solo (optimized)        Raw     0.845


## Final results